In [1]:
from csrio_image2biomass.configs.settings import AUGUMENTED_DATA_DIR
import polars as pl
train = pl.read_csv(AUGUMENTED_DATA_DIR / "train.csv")
test = pl.read_csv(AUGUMENTED_DATA_DIR / "test.csv")
train.sort("image_path").head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hvflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_vflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import numpy as np
import os
from typing import Dict, Any, Tuple

class BiomassDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, img_dir: str, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        item: Dict[str, Any] = self.dataframe.row(idx, named=True)
        img_name = os.path.join(self.img_dir, item['image_path'])
        image = Image.open(img_name).convert('RGB')
        labels = np.array([item['Dry_Clover_g'], item['Dry_Dead_g'], item['Dry_Green_g'], item['Dry_Total_g'], item['GDM_g']], dtype=np.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels
    
train_dataset = BiomassDataset(dataframe=train, img_dir=str(AUGUMENTED_DATA_DIR), transform=transforms.ToTensor())
test_dataset = BiomassDataset(dataframe=test, img_dir=str(AUGUMENTED_DATA_DIR), transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

for batch in train_loader:
    images, labels = batch
    print(f"Image batch shape: {images.size()}")
    print(f"Label batch shape: {labels.size()}")
    break


Using device: cuda
Image batch shape: torch.Size([32, 3, 1000, 2000])
Label batch shape: torch.Size([32, 5])


In [3]:
model = models.mobilenet_v2(weights=True)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 5)
model = model.to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/ravikumar/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


/home/ravikumar/kaggle/csrio-image2biomass/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 13.6M/13.6M [00:01<00:00, 8.73MB/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.91 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 10.26 GiB is allocated by PyTorch, and 20.86 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)